NeoKarir Job Match — Optimized v3
Target: Val Accuracy > 92%

Perubahan dari v2:
1. DATA: Template lebih bervariasi (3x lipat), tambah "partial match" case,
         hard negative mining berbasis role cluster, augmentasi teks (swap/drop kata)
2. ENCODER: CNN parallel branch (1D conv trigram) + BiLSTM → richer local features
3. SIMILARITY: cosine_sim di-tile ke full vector, tambah squared_diff
4. TRAINING: Warmup manual yang benar (linear ramp), mixup alpha=0.1,
             threshold tuning post-training via PR curve
5. ARSITEKTUR: Bottleneck projection sebelum similarity head, gated fusion

In [1]:
import os, random, re, pickle
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, regularizers
from sklearn.metrics import (classification_report, roc_auc_score,
                             accuracy_score, precision_recall_curve)
from sklearn.utils import class_weight as sk_class_weight

# ── Reproducibility ──────────────────────────────────────────────────
SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print(f"TensorFlow version : {tf.__version__}")
print(f"GPU available      : {bool(tf.config.list_physical_devices('GPU'))}")

2026-05-17 05:41:57.766042: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1778996517.968882      24 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1778996518.027363      24 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1778996518.503310      24 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1778996518.503359      24 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1778996518.503362      24 computation_placer.cc:177] computation placer alr

TensorFlow version : 2.19.0
GPU available      : True


In [2]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# HYPERPARAMETERS
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
MAX_TOKENS        = 20000
SEQUENCE_LENGTH   = 80        # sedikit lebih panjang untuk CV yang richer
EMBEDDING_DIM     = 96        # naik dari 64
LSTM_UNITS        = 96        # per direction; BiLSTM output = 192
CNN_FILTERS       = 64        # parallel CNN branch
CNN_KERNELS       = [2, 3, 4] # tri-gram, quad-gram, penta-gram
ATT_HEADS         = 4
ATT_KEY_DIM       = 32
PROJ_DIM          = 192       # bottleneck projection dim sebelum similarity
DENSE_UNITS       = 384       # naik dari 256
BATCH_SIZE        = 64
EPOCHS            = 20
WARMUP_EPOCHS     = 6
INIT_LR           = 8e-4
MIN_LR            = 5e-6
DROPOUT_RATE      = 0.30      # turun sedikit dari 0.35 (model butuh lebih belajar)
L2_REG            = 5e-5      # turun dari 1e-4
VALIDATION_SPLIT  = 0.15
MIXUP_ALPHA       = 0.1       # ringan, hanya untuk train
LABEL_SMOOTHING   = 0.04

In [3]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# CELL 1: DATA GENERATION (DIPERKAYA)
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
print("\n" + "="*60)
print("CELL 1 — Data Loading & Augmentation (v3)")
print("="*60)

roles = [
    "Software Engineer","Backend Developer","Frontend Developer","Fullstack Developer",
    "Data Scientist","Data Engineer","Data Analyst","Machine Learning Engineer",
    "AI Engineer","DevOps Engineer","Cloud Architect","Site Reliability Engineer",
    "QA Automation","QA Manual","UI/UX Designer","Product Manager",
    "Scrum Master","Mobile Developer","iOS Developer","Android Developer",
    "Game Developer","Security Analyst","Penetration Tester","Blockchain Developer",
    "Database Administrator","Network Engineer","System Administrator",
    "Solutions Architect","Platform Engineer","MLOps Engineer","NLP Engineer",
]

# Cluster role berdasarkan domain — untuk hard negative yang lebih realistis
ROLE_CLUSTERS = {
    "backend":   ["Backend Developer","Software Engineer","Fullstack Developer","Database Administrator"],
    "frontend":  ["Frontend Developer","UI/UX Designer","Mobile Developer","iOS Developer","Android Developer"],
    "data":      ["Data Scientist","Data Analyst","Data Engineer","Machine Learning Engineer",
                  "AI Engineer","NLP Engineer","MLOps Engineer"],
    "infra":     ["DevOps Engineer","Cloud Architect","Site Reliability Engineer","Network Engineer",
                  "System Administrator","Platform Engineer"],
    "security":  ["Security Analyst","Penetration Tester"],
    "product":   ["Product Manager","Scrum Master"],
    "game":      ["Game Developer"],
    "blockchain":["Blockchain Developer"],
}

# ================= AI vs NON-AI DOMAIN =================

AI_ROLES = [
    "AI Engineer", "Machine Learning Engineer",
    "Data Scientist", "NLP Engineer", "MLOps Engineer"
]

AI_KEYWORDS = [
    "machine learning", "deep learning", "nlp",
    "computer vision", "rag", "tensorflow",
    "pytorch", "ml ops", "model deployment"
]

NON_AI_ROLES = [
    "Frontend Developer", "Mobile Developer",
    "UI/UX Designer", "Cyber Security Analyst",
    "IT Support", "Network Engineer"
]

def same_cluster(r1, r2):
    for cluster in ROLE_CLUSTERS.values():
        if r1 in cluster and r2 in cluster:
            return True
    return False

languages  = ["Python","Java","C++","C#","JavaScript","TypeScript","Go","Golang",
               "Rust","Ruby","PHP","Swift","Kotlin","Dart","R","Scala","SQL",
               "NoSQL","Bash","Shell","MATLAB","Solidity","Haskell","Elixir"]
frameworks = ["React","Angular","Vue.js","Next.js","Node.js","Express","NestJS",
               "Django","Flask","FastAPI","Spring Boot","Laravel","TensorFlow","PyTorch",
               "Keras","Scikit-learn","Pandas","NumPy","XGBoost","HuggingFace","Flutter",
               "React Native","Tailwind CSS","Bootstrap","gRPC","GraphQL","Gin","Echo",
               "Fiber","Actix","Phoenix","Rails","Sinatra","Ktor","Compose","SwiftUI"]
tools      = ["Docker","Kubernetes","Jenkins","GitLab CI","GitHub Actions","Terraform",
               "Ansible","AWS","GCP","Azure","Linux","Nginx","Kafka","Redis","Elasticsearch",
               "Prometheus","Grafana","MySQL","PostgreSQL","MongoDB","Cassandra","Firebase",
               "Git","Jira","Figma","Tableau","Power BI","Airflow","Snowflake","BigQuery",
               "Databricks","MLflow","DVC","Weights & Biases","Sentry","DataDog","Splunk",
               "RabbitMQ","Celery","Spark","Flink","dbt","Looker","Metabase"]
soft_skills = ["kepemimpinan","komunikasi efektif","kolaborasi tim","pemecahan masalah",
                "berpikir kritis","agile mindset","scrum","kanban","manajemen waktu",
                "mentoring junior","adaptabilitas","ownership","analytical thinking",
                "stakeholder management","cross-functional collaboration"]
action_verbs = ["merancang","mengembangkan","mengimplementasikan","mendeploy","memaintain",
                 "menskalakan","mengoptimasi","mengintegrasikan","mengtest","mendebug",
                 "menganalisis","mengotomasi","merefaktor","memimpin","membangun",
                 "architected","designed","developed","deployed","scaled","optimized",
                 "led","built","maintained","integrated","automated"]
levels   = ["Junior","Mid-level","Senior","Lead","Principal","Manager","Fresh Graduate","Intern"]
degrees  = ["S1 Informatika","S1 Sistem Informasi","S1 Ilmu Komputer",
            "S1 Teknik Elektro","S1 Statistika","Bootcamp Graduate",
            "S2 Computer Science","S1 Matematika","D3 Teknik Komputer"]
companies = ["startup","perusahaan teknologi","fintech","e-commerce","konsultan IT",
             "BUMN teknologi","unicorn","decacorn","SaaS company"]

# Template CV lebih banyak dan bervariasi
templates_cv = [
    "{level} {role} dengan pengalaman {exp} tahun. Menguasai {tech1}, {tech2}, dan {tech3}. Berpengalaman {action} sistem menggunakan {tech4}.",
    "Lulusan {degree} mencari posisi {role}. Portfolio mencakup {tech1}, {tech2}, {tech3}. Soft skill: {soft}.",
    "{exp} tahun sebagai {role}. Stack utama: {tech1}, {tech2}, {tech3}, {tech4}. Pernah {action} proyek besar. {soft}.",
    "Saya seorang {role} yang fokus pada {tech1} dan {tech2}. Pernah {action} aplikasi dengan {tech3} dan {tech4}.",
    "Experienced {role} ({level}) with {exp} years experience. Core skills: {tech1}, {tech2}, {tech3}. Familiar with {tech4}.",
    "{role} dengan keahlian utama {tech1}, {tech2}. Background: {action}, {tech3}, {tech4}. Pendidikan: {degree}.",
    "Sebagai {level} {role} di {company} selama {exp} tahun, saya telah {action} berbagai sistem menggunakan {tech1}, {tech2}, {tech3}.",
    "{role} passionate tentang {tech1}. Memiliki pengalaman {exp} tahun menggunakan {tech2}, {tech3}, {tech4}. Soft skill utama: {soft}.",
    "Background {degree}, kini bekerja sebagai {role}. Tech stack sehari-hari: {tech1}, {tech2}, {tech3}. Pernah {action} di {company}.",
    "{exp}+ years of experience as {role}. Proficient in {tech1}, {tech2}, {tech3}, {tech4}. Strong {soft}.",
    "Seorang {level} {role} yang berpengalaman {action} platform skala besar. Skill: {tech1}, {tech2}, {tech3}.",
    "CV: {role} | {level} | {exp} tahun | {tech1} | {tech2} | {tech3} | {tech4} | {soft}.",
    "Mencari posisi {role} sebagai {level}. Keahlian teknis: {tech1}, {tech2}, {tech3}. Lulusan {degree} dari {company}.",
    "{role} with hands-on experience in {tech1} and {tech2}. Also worked with {tech3}, {tech4}. {exp} years total.",
    "Saya {role} yang gemar {action} solusi dengan {tech1} dan {tech2}. Background: {degree}, {exp} tahun experience.",
]

templates_job = [
    "Dibutuhkan {level} {role}. Syarat: {exp}+ tahun pengalaman, mahir {tech1} dan {tech2}. Nice to have: {tech3}.",
    "Hiring {role}! Wajib: {tech1}, {tech2}, {tech3}. Diutamakan yang pernah {action} menggunakan {tech4}.",
    "Lowongan {role} ({level}). Stack kami: {tech1}, {tech2}, {tech3}. Soft skill dibutuhkan: {soft}.",
    "We are looking for a {level} {role} with strong {tech1} and {tech2} skills. Experience in {tech3} is a plus.",
    "Posisi {role} untuk kandidat {level}. Harus bisa {tech1}, {tech2}, {tech3}. Familiar {tech4} nilai plus.",
    "Cari {role} berpengalaman {exp} tahun. Skill wajib: {tech1}, {tech2}. Diutamakan: {tech3}, {tech4}.",
    "Join tim kami sebagai {role}! Kami butuh kandidat dengan {exp}+ tahun exp di {tech1}, {tech2}, {tech3}.",
    "Posisi {level} {role} di {company}. Requirements: {tech1}, {tech2}, {tech3}, {tech4}. Soft skill: {soft}.",
    "We need a talented {role} who can {action} complex systems. Must know {tech1}, {tech2}. Plus: {tech3}.",
    "Open position: {role}. Min {exp} tahun pengalaman. Wajib: {tech1}, {tech2}. Optional: {tech3}, {tech4}.",
    "Rekrutmen {role} ({level}). Kualifikasi teknis: {tech1}, {tech2}, {tech3}. Pendidikan min {degree}.",
    "Job desc: {role} akan {action} sistem {company} menggunakan {tech1}, {tech2}, {tech3}. Exp: {exp}+ tahun.",
    "{role} needed ASAP. Stack: {tech1}, {tech2}, {tech3}, {tech4}. {level} preferred. Min {exp} years.",
    "Bergabunglah sebagai {role} di {company}! Skill utama: {tech1}, {tech2}. Bonus: {tech3}, {tech4}.",
    "Requirement: {level} {role}. Technical: {tech1}, {tech2}, {tech3}. Culture fit: {soft}.",
]

def _sample(pool, n):
    n = min(n, len(pool))
    return random.sample(pool, n)

def fill_template(tpl, role, level, exp, degree, action, soft, techs, company=""):
    t = list(techs) + [techs[0]] * 6
    return tpl.format(
        level=level, role=role, exp=exp, degree=degree,
        action=action, soft=soft, company=company or "startup",
        tech1=t[0], tech2=t[1], tech3=t[2], tech4=t[3], tech5=t[4]
    )

def augment_text(text, drop_prob=0.08, swap_prob=0.05):
    """Augmentasi ringan: random word drop dan swap."""
    words = text.split()
    if len(words) < 5:
        return text
    # Random drop
    words = [w for w in words if random.random() > drop_prob]
    # Random adjacent swap
    for i in range(len(words) - 1):
        if random.random() < swap_prob:
            words[i], words[i+1] = words[i+1], words[i]
    return " ".join(words)

ALL_TECH = languages + frameworks + tools

def generate_synthetic_data(num_iterations):
    data = []
    for _ in range(num_iterations):
        role   = random.choice(roles)
        level  = random.choice(levels)
        exp    = str(random.randint(0, 12))
        degree = random.choice(degrees)
        action = random.choice(action_verbs)
        soft   = random.choice(soft_skills)
        company = random.choice(companies)

        # ── Kasus 1: MATCH sempurna (label=1) ────────────────────────
        # Overlap tech pool tinggi (6-8 shared dari 14)
        tech_pool = _sample(ALL_TECH, 14)
        n_shared  = random.randint(4, 6)
        shared    = _sample(tech_pool, n_shared)
        extra_cv  = _sample([t for t in tech_pool if t not in shared], 2)
        extra_job = _sample([t for t in tech_pool if t not in shared and t not in extra_cv], 2)
        techs_cv  = shared + extra_cv
        techs_job = shared + extra_job

        cv_text  = fill_template(random.choice(templates_cv), role, level, exp,
                                 degree, action, soft, techs_cv, company)
        job_text = fill_template(random.choice(templates_job), role, level, exp,
                                 degree, action, soft, techs_job, company)
        data.append({"cv_text": cv_text, "job_text": job_text, "label": 1})

        # ── Kasus 1b: PARTIAL MATCH (label=1) ────────────────────────
        # Role sama, tech overlap sedang (2-3 shared), level berbeda
        other_level = random.choice([l for l in levels if l != level])
        partial_shared = _sample(tech_pool, 2)
        techs_cv_p  = partial_shared + _sample([t for t in tech_pool if t not in partial_shared], 3)
        techs_job_p = partial_shared + _sample([t for t in ALL_TECH if t not in partial_shared], 3)
        cv_p  = fill_template(random.choice(templates_cv), role, level, exp,
                              degree, action, soft, techs_cv_p, company)
        job_p = fill_template(random.choice(templates_job), role, other_level, exp,
                              degree, action, soft, techs_job_p, company)
        # Augment salah satu
        if random.random() > 0.5:
            cv_p = augment_text(cv_p)
        data.append({"cv_text": cv_p, "job_text": job_p, "label": 1})

        # ── Kasus 2: HARD NEGATIVE — same role, diff tech (label=0) ──
        # Tech benar-benar berbeda (out of pool)
        other_tech = [t for t in ALL_TECH if t not in tech_pool]
        neg_pool   = _sample(other_tech, 8) if len(other_tech) >= 8 else other_tech + other_tech
        techs_neg  = _sample(neg_pool, 5)
        job_neg    = fill_template(random.choice(templates_job), role, level, exp,
                                   degree, action, soft, techs_neg, company)
        data.append({"cv_text": cv_text, "job_text": job_neg, "label": 0})

        # ── Kasus 3: MISMATCH — role beda cluster, tech beda (label=0) ──
        other_role = random.choice([r for r in roles
                                    if r != role and not same_cluster(r, role)])
        if not other_role:
            other_role = random.choice([r for r in roles if r != role])
        techs_rand = _sample(neg_pool, 5)
        job_rand   = fill_template(random.choice(templates_job), other_role, level, exp,
                                   degree, action, soft, techs_rand, company)
        data.append({"cv_text": cv_text, "job_text": job_rand, "label": 0})

        # ── Kasus 4: NEAR MISS — role sama cluster, tech beda (label=0) ──
        # Paling sulit: role mirip tapi skill tidak overlap
        cluster_roles = []
        for c in ROLE_CLUSTERS.values():
            if role in c:
                cluster_roles = [r for r in c if r != role]
                break
        if cluster_roles:
            near_role = random.choice(cluster_roles)
            techs_near = _sample([t for t in ALL_TECH if t not in tech_pool], 5)
            job_near   = fill_template(random.choice(templates_job), near_role, level, exp,
                                       degree, action, soft, techs_near, company)
            data.append({"cv_text": cv_text, "job_text": job_near, "label": 0})

        # ── Kasus 5: AI vs NON-AI (HARD NEGATIVE DOMAIN) ──
        if role in AI_ROLES:
            non_ai_role = random.choice(NON_AI_ROLES)

            techs_non_ai = _sample([t for t in ALL_TECH if t not in tech_pool], 5)

            job_non_ai = fill_template(
                random.choice(templates_job),
                non_ai_role,
                level,
                exp,
                degree,
                action,
                soft,
                techs_non_ai,
                company
            )

            data.append({
                "cv_text": cv_text,
                "job_text": job_non_ai,
                "label": 0
            })


        # ── Kasus 6: NON-AI CV vs AI JOB (REVERSE NEGATIVE) ──
        if role not in AI_ROLES:
            ai_role = random.choice(AI_ROLES)

            ai_skills = _sample(AI_KEYWORDS, 3)
            techs_ai = ai_skills + _sample(ALL_TECH, 2)

            job_ai = fill_template(
                random.choice(templates_job),
                ai_role,
                level,
                exp,
                degree,
                action,
                soft,
                techs_ai,
                company
            )

            data.append({
                "cv_text": cv_text,
                "job_text": job_ai,
                "label": 0
            })

    return pd.DataFrame(data)

# Load + augment
df_asli = pd.read_csv("/kaggle/input/datasets/laventiliz/job-match-v4/job_match_v4.csv")
print(f"Original data: {len(df_asli)} rows")

# Generate lebih banyak: 5000 iterasi × ~4-5 samples = ±20k-25k rows sintetis
print("Generating synthetic rows (5000 iterations)...")
df_sintetis = generate_synthetic_data(5000)

df = pd.concat([df_asli, df_sintetis], ignore_index=True)
df = df.dropna(subset=["cv_text", "job_text", "label"])
df = df.sample(frac=1, random_state=SEED).reset_index(drop=True)
df["label"] = df["label"].astype(int)

print(f"Total rows: {len(df)}")
print(f"Label dist:\n{df['label'].value_counts().to_string()}")


CELL 1 — Data Loading & Augmentation (v3)
Original data: 800 rows
Generating synthetic rows (5000 iterations)...
Total rows: 30002
Label dist:
label
0    19602
1    10400


In [4]:
df.to_csv("job_match_score_v4_extended2.csv")

In [5]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# CELL 2: PREPROCESSING
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
print("\n" + "="*60)
print("CELL 2 — Text Vectorization & tf.data Pipeline")
print("="*60)

vectorizer = layers.TextVectorization(
    max_tokens=MAX_TOKENS,
    output_mode="int",
    output_sequence_length=SEQUENCE_LENGTH,
    standardize="lower_and_strip_punctuation",
    name="shared_text_vectorizer",
)
all_texts = pd.concat([df["cv_text"], df["job_text"]]).astype(str).tolist()
vectorizer.adapt(all_texts)

vocab_size = len(vectorizer.get_vocabulary())
print(f"Vocab size: {vocab_size:,}")

cv_sequences  = vectorizer(tf.constant(df["cv_text"].astype(str).tolist()))
job_sequences = vectorizer(tf.constant(df["job_text"].astype(str).tolist()))
labels        = tf.constant(df["label"].values, dtype=tf.float32)

n_total = len(labels)
n_val   = int(n_total * VALIDATION_SPLIT)
n_train = n_total - n_val

indices   = tf.random.shuffle(tf.range(n_total), seed=SEED)
train_idx = indices[:n_train]
val_idx   = indices[n_train:]

cv_train,  cv_val  = tf.gather(cv_sequences, train_idx),  tf.gather(cv_sequences, val_idx)
job_train, job_val = tf.gather(job_sequences, train_idx), tf.gather(job_sequences, val_idx)
y_train,   y_val   = tf.gather(labels, train_idx),        tf.gather(labels, val_idx)

print(f"Train: {n_train} | Val: {n_val}")

# Class weights
n_pos = int(tf.reduce_sum(y_train).numpy())
n_neg = n_train - n_pos
cw_neg = n_train / (2 * n_neg)
cw_pos = n_train / (2 * n_pos)
print(f"Class weights: {{0: {cw_neg:.3f}, 1: {cw_pos:.3f}}}")

def make_dataset(cv, job, y, shuffle=False):
    ds = tf.data.Dataset.from_tensor_slices(((cv, job), y))
    if shuffle:
        ds = ds.shuffle(buffer_size=min(n_train, 30000), seed=SEED,
                        reshuffle_each_iteration=True)
    return ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

train_ds = make_dataset(cv_train, job_train, y_train, shuffle=True)
val_ds   = make_dataset(cv_val,   job_val,   y_val,   shuffle=False)
print(f"Train batches: {len(train_ds)} | Val batches: {len(val_ds)}")


CELL 2 — Text Vectorization & tf.data Pipeline


I0000 00:00:1778996545.740850      24 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13757 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1778996545.746839      24 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13757 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


Vocab size: 542
Train: 25502 | Val: 4500
Class weights: {0: 0.765, 1: 1.444}
Train batches: 399 | Val batches: 71


In [6]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# CELL 3: MODEL ARCHITECTURE
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
print("\n" + "="*60)
print("CELL 3 — Model Architecture (CNN + BiLSTM + MHA Siamese)")
print("="*60)

class DistanceLayer(layers.Layer):
    def call(self, a, b): return tf.abs(a - b)
    def get_config(self): return super().get_config()

class SquaredDistanceLayer(layers.Layer):
    def call(self, a, b): return tf.square(a - b)
    def get_config(self): return super().get_config()

class CosineTileLayer(layers.Layer):
    """Cosine similarity di-tile ke dimensi vektor agar dimensinya konsisten."""
    def __init__(self, dim, **kwargs):
        super().__init__(**kwargs)
        self.dim = dim
    def call(self, a, b):
        a_n = tf.nn.l2_normalize(a, axis=-1)
        b_n = tf.nn.l2_normalize(b, axis=-1)
        cos = tf.reduce_sum(a_n * b_n, axis=-1, keepdims=True)  # (B,1)
        return tf.tile(cos, [1, self.dim])                        # (B, dim)
    def get_config(self):
        cfg = super().get_config()
        cfg["dim"] = self.dim
        return cfg

def build_encoder(name="shared_encoder"):
    """
    CNN branch (parallel conv kernels) + BiLSTM branch + MHA
    → concat → projection → output
    """
    reg = regularizers.l2(L2_REG)

    inp = keras.Input(shape=(SEQUENCE_LENGTH,), dtype=tf.int32, name=f"{name}_input")

    emb = layers.Embedding(
        input_dim=vocab_size,
        output_dim=EMBEDDING_DIM,
        mask_zero=True,
        embeddings_regularizer=reg,
        name=f"{name}_emb",
    )(inp)
    emb_drop = layers.SpatialDropout1D(DROPOUT_RATE * 0.6,
                                       name=f"{name}_emb_drop")(emb)

    # ── CNN Branch: parallel conv → GlobalMaxPool ─────────────────
    cnn_pools = []
    for k in CNN_KERNELS:
        c = layers.Conv1D(CNN_FILTERS, k, activation="relu", padding="same",
                          kernel_regularizer=reg,
                          name=f"{name}_conv{k}")(emb_drop)
        c = layers.GlobalMaxPooling1D(name=f"{name}_maxpool{k}")(c)
        cnn_pools.append(c)
    cnn_out = layers.Concatenate(name=f"{name}_cnn_cat")(cnn_pools)     # (B, 192)
    cnn_out = layers.Dense(LSTM_UNITS, activation="relu",
                           kernel_regularizer=reg,
                           name=f"{name}_cnn_proj")(cnn_out)             # (B, 96)
    cnn_out = layers.BatchNormalization(name=f"{name}_cnn_bn")(cnn_out)

    # ── BiLSTM Branch ─────────────────────────────────────────────
    x = layers.Bidirectional(
        layers.LSTM(LSTM_UNITS, dropout=DROPOUT_RATE * 0.4,
                    return_sequences=True, kernel_regularizer=reg),
        name=f"{name}_bilstm",
    )(emb_drop)

    # Multi-Head Attention + residual + norm
    attn = layers.MultiHeadAttention(
        num_heads=ATT_HEADS, key_dim=ATT_KEY_DIM,
        dropout=DROPOUT_RATE * 0.25,
        name=f"{name}_mha",
    )(x, x)
    x = layers.Add(name=f"{name}_res")([x, attn])
    x = layers.LayerNormalization(name=f"{name}_ln")(x)
    lstm_out = layers.GlobalAveragePooling1D(name=f"{name}_gap")(x)     # (B, 192)

    lstm_out = layers.Dense(LSTM_UNITS, activation="relu",
                            kernel_regularizer=reg,
                            name=f"{name}_lstm_proj")(lstm_out)          # (B, 96)
    lstm_out = layers.BatchNormalization(name=f"{name}_lstm_bn")(lstm_out)

    # ── Merge CNN + LSTM ──────────────────────────────────────────
    merged = layers.Concatenate(name=f"{name}_merge")([cnn_out, lstm_out])  # (B, 192)
    merged = layers.Dropout(DROPOUT_RATE, name=f"{name}_drop")(merged)

    # Bottleneck projection
    out = layers.Dense(PROJ_DIM, activation="relu",
                       kernel_regularizer=reg,
                       name=f"{name}_final_proj")(merged)
    out = layers.BatchNormalization(name=f"{name}_final_bn")(out)

    return keras.Model(inputs=inp, outputs=out, name=name)


def build_siamese_model():
    encoder = build_encoder("shared_encoder")

    cv_input  = keras.Input(shape=(SEQUENCE_LENGTH,), dtype=tf.int32, name="cv_input")
    job_input = keras.Input(shape=(SEQUENCE_LENGTH,), dtype=tf.int32, name="job_input")

    cv_vec  = encoder(cv_input)
    job_vec = encoder(job_input)

    # 4 similarity signals
    abs_diff   = DistanceLayer(name="abs_diff")(cv_vec, job_vec)        # (B, PROJ_DIM)
    sq_diff    = SquaredDistanceLayer(name="sq_diff")(cv_vec, job_vec)  # (B, PROJ_DIM)
    hadamard   = layers.Multiply(name="hadamard")([cv_vec, job_vec])    # (B, PROJ_DIM)
    cos_tile   = CosineTileLayer(PROJ_DIM, name="cos_tile")(cv_vec, job_vec)  # (B, PROJ_DIM)

    # Concat: 4 × PROJ_DIM
    x = layers.Concatenate(name="concat_signals")([abs_diff, sq_diff, hadamard, cos_tile])

    # Deep MLP classifier
    reg = regularizers.l2(L2_REG)

    x = layers.Dense(DENSE_UNITS, activation="relu",
                     kernel_regularizer=reg, name="dense_1")(x)
    x = layers.BatchNormalization(name="bn_1")(x)
    x = layers.Dropout(DROPOUT_RATE, name="drop_1")(x)

    x = layers.Dense(DENSE_UNITS // 2, activation="relu",
                     kernel_regularizer=reg, name="dense_2")(x)
    x = layers.BatchNormalization(name="bn_2")(x)
    x = layers.Dropout(DROPOUT_RATE * 0.6, name="drop_2")(x)

    x = layers.Dense(DENSE_UNITS // 4, activation="relu",
                     kernel_regularizer=reg, name="dense_3")(x)
    x = layers.BatchNormalization(name="bn_3")(x)
    x = layers.Dropout(DROPOUT_RATE * 0.4, name="drop_3")(x)

    output = layers.Dense(1, activation="sigmoid", name="match_score")(x)

    return keras.Model(inputs=[cv_input, job_input], outputs=output,
                       name="siamese_job_match_v3")


model = build_siamese_model()
model.summary(expand_nested=False)


CELL 3 — Model Architecture (CNN + BiLSTM + MHA Siamese)


/usr/local/lib/python3.12/dist-packages/keras/src/layers/layer.py:965: UserWarning: Layer 'shared_encoder_conv2' (of type Conv1D) was passed an input with a mask attached to it. However, this layer does not support masking and will therefore destroy the mask information. Downstream layers will not see the mask.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/keras/src/layers/layer.py:965: UserWarning: Layer 'shared_encoder_conv3' (of type Conv1D) was passed an input with a mask attached to it. However, this layer does not support masking and will therefore destroy the mask information. Downstream layers will not see the mask.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/keras/src/layers/layer.py:965: UserWarning: Layer 'shared_encoder_conv4' (of type Conv1D) was passed an input with a mask attached to it. However, this layer does not support masking and will therefore destroy the mask information. Downstream layers will not see the mask.
  warnings.warn(


Model: "siamese_job_match_v3"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ cv_input            │ (None, 80)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ job_input           │ (None, 80)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ shared_encoder      │ (None, 192)       │    430,656 │ cv_input[0][0],   │
│ (Functional)        │                   │            │ job_input[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ abs_diff            │ (None, 192)       │          0 │ shared_encoder[0… │
│ (DistanceLayer)     │                   │            │ shared_encoder[1… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ sq_diff             │ (None, 192)       │          0 │ shared_encoder[0… │
│ (SquaredDistanceLa… │                   │            │ shared_encoder[1… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ hadamard (Multiply) │ (None, 192)       │          0 │ shared_encoder[0… │
│                     │                   │            │ shared_encoder[1… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ cos_tile            │ (None, 192)       │          0 │ shared_encoder[0… │
│ (CosineTileLayer)   │                   │            │ shared_encoder[1… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concat_signals      │ (None, 768)       │          0 │ abs_diff[0][0],   │
│ (Concatenate)       │                   │            │ sq_diff[0][0],    │
│                     │                   │            │ hadamard[0][0],   │
│                     │                   │            │ cos_tile[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 384)       │    295,296 │ concat_signals[0… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bn_1                │ (None, 384)       │      1,536 │ dense_1[0][0]     │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ drop_1 (Dropout)    │ (None, 384)       │          0 │ bn_1[0][0]        │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_2 (Dense)     │ (None, 192)       │     73,920 │ drop_1[0][0]      │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bn_2                │ (None, 192)       │        768 │ dense_2[0][0]     │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ drop_2 (Dropout)    │ (None, 192)       │          0 │ bn_2[0][0]        │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_3 (Dense)     │ (None, 96)        │     18,528 │ drop_2[0][0]      │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bn_3                │ (None, 96)        │        384 │ dense_3[0][0]     │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ drop_3 (Dropout)    │ (None, 96)        │          0 │ bn_3[0][0]        │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ match_score (Dense) │ (None, 1)         │         97 │ drop_3[0][0]      │
└─────────────────────┴───────────────────┴────────────┴─────────────────

 Total params: 821,185 (3.13 MB)

 Trainable params: 819,073 (3.12 MB)

 Non-trainable params: 2,112 (8.25 KB)

In [7]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# CELL 4: TRAINING — Warmup Manual + Cosine Decay + Mixup
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
print("\n" + "="*60)
print("CELL 4 — Training (Warmup + Cosine Decay + Mixup)")
print("="*60)

total_steps  = EPOCHS * len(train_ds)
warmup_steps = WARMUP_EPOCHS * len(train_ds)

class WarmupCosineDecay(tf.keras.optimizers.schedules.LearningRateSchedule):
    """Linear warmup kemudian cosine decay ke MIN_LR."""
    def __init__(self, init_lr, min_lr, warmup_steps, total_steps):
        super().__init__()
        self.init_lr      = float(init_lr)
        self.min_lr       = float(min_lr)
        self.warmup_steps = float(warmup_steps)
        self.total_steps  = float(total_steps)

    def __call__(self, step):
        step = tf.cast(step, tf.float32)
        # Warmup phase
        warmup_lr = self.init_lr * (step / self.warmup_steps)
        # Cosine decay phase
        progress  = (step - self.warmup_steps) / (self.total_steps - self.warmup_steps)
        progress  = tf.clip_by_value(progress, 0.0, 1.0)
        cosine_lr = self.min_lr + 0.5 * (self.init_lr - self.min_lr) * (
                        1.0 + tf.math.cos(np.pi * progress))
        return tf.where(step < self.warmup_steps, warmup_lr, cosine_lr)

    def get_config(self):
        return {
            "init_lr": self.init_lr, "min_lr": self.min_lr,
            "warmup_steps": self.warmup_steps, "total_steps": self.total_steps,
        }

lr_schedule = WarmupCosineDecay(INIT_LR, MIN_LR, warmup_steps, total_steps)
optimizer   = tf.keras.optimizers.Adam(learning_rate=lr_schedule, clipnorm=1.0)
loss_fn     = tf.keras.losses.BinaryCrossentropy(label_smoothing=LABEL_SMOOTHING)

train_acc_metric  = tf.keras.metrics.BinaryAccuracy()
val_acc_metric    = tf.keras.metrics.BinaryAccuracy()
train_loss_metric = tf.keras.metrics.Mean()
val_loss_metric   = tf.keras.metrics.Mean()

# ── Mixup augmentation ────────────────────────────────────────────────
def mixup_batch(cv_b, job_b, y_b, alpha=MIXUP_ALPHA):
    """Lightweight mixup: interpolasi label saja (feature mixing untuk int tensor
    tidak praktis, jadi kita hanya mix label dengan lambda kecil)."""
    if alpha <= 0 or random.random() > 0.3:  # 30% chance per batch
        return cv_b, job_b, y_b
    lam = np.random.beta(alpha, alpha)
    lam = max(lam, 1 - lam)  # pastikan lam >= 0.5
    idx = tf.random.shuffle(tf.range(tf.shape(y_b)[0]))
    y_b_mix = lam * tf.cast(y_b, tf.float32) + (1 - lam) * tf.cast(
              tf.gather(y_b, idx), tf.float32)
    return cv_b, job_b, y_b_mix

@tf.function
def val_step(inputs, y_b):
    preds = model(inputs, training=False)
    loss  = loss_fn(y_b, preds)
    val_acc_metric.update_state(y_b, preds)
    val_loss_metric.update_state(loss)

# ── Training Loop ─────────────────────────────────────────────────────
best_val_acc = 0.0
patience     = 10
no_improve   = 0
history      = {"loss": [], "accuracy": [], "val_loss": [], "val_accuracy": []}
cw_pos_t     = tf.constant(cw_pos, dtype=tf.float32)
cw_neg_t     = tf.constant(cw_neg, dtype=tf.float32)

print(f"\nTraining for up to {EPOCHS} epochs (patience={patience})...\n")

for epoch in range(1, EPOCHS + 1):
    train_acc_metric.reset_state()
    train_loss_metric.reset_state()
    val_acc_metric.reset_state()
    val_loss_metric.reset_state()

    # Train
    for (cv_b, job_b), y_b in train_ds:
        cv_b, job_b, y_b = mixup_batch(cv_b, job_b, y_b)

        weights = tf.where(
            tf.equal(tf.round(y_b), 1.0),
            tf.fill(tf.shape(y_b), cw_pos_t),
            tf.fill(tf.shape(y_b), cw_neg_t),
        )
        with tf.GradientTape() as tape:
            preds = model([cv_b, job_b], training=True)
            loss  = tf.reduce_mean(loss_fn(y_b, preds) * weights)
            loss += sum(model.losses)
        grads = tape.gradient(loss, model.trainable_variables)
        grads, _ = tf.clip_by_global_norm(grads, 1.0)
        optimizer.apply_gradients(zip(grads, model.trainable_variables))
        train_acc_metric.update_state(tf.round(y_b), preds)
        train_loss_metric.update_state(loss)

    # Validate
    for (cv_b, job_b), y_b in val_ds:
        val_step((cv_b, job_b), y_b)

    tr_loss = train_loss_metric.result().numpy()
    tr_acc  = train_acc_metric.result().numpy()
    vl_loss = val_loss_metric.result().numpy()
    vl_acc  = val_acc_metric.result().numpy()

    history["loss"].append(tr_loss)
    history["accuracy"].append(tr_acc)
    history["val_loss"].append(vl_loss)
    history["val_accuracy"].append(vl_acc)

    marker = ""
    if vl_acc > best_val_acc:
        best_val_acc = vl_acc
        model.save_weights("best_weights_v3.weights.h5")
        no_improve = 0
        marker = " ← BEST ✅"
    else:
        no_improve += 1

    print(f"Epoch {epoch:3d}/{EPOCHS} | "
          f"loss={tr_loss:.4f} acc={tr_acc:.4f} | "
          f"val_loss={vl_loss:.4f} val_acc={vl_acc:.4f}{marker}")

    if no_improve >= patience:
        print(f"\nEarly stopping setelah {patience} epoch tanpa improvement.")
        break


CELL 4 — Training (Warmup + Cosine Decay + Mixup)

Training for up to 20 epochs (patience=10)...



I0000 00:00:1778996551.703634      71 cuda_dnn.cc:529] Loaded cuDNN version 91002


Epoch   1/20 | loss=0.9285 acc=0.5359 | val_loss=0.8041 val_acc=0.4698 ← BEST ✅
Epoch   2/20 | loss=0.7510 acc=0.6505 | val_loss=0.9923 val_acc=0.5389 ← BEST ✅
Epoch   3/20 | loss=0.5716 acc=0.7818 | val_loss=0.8112 val_acc=0.7051 ← BEST ✅
Epoch   4/20 | loss=0.4508 acc=0.8692 | val_loss=0.6437 val_acc=0.7887 ← BEST ✅
Epoch   5/20 | loss=0.3801 acc=0.9088 | val_loss=0.4779 val_acc=0.8264 ← BEST ✅
Epoch   6/20 | loss=0.3399 acc=0.9308 | val_loss=0.3390 val_acc=0.8760 ← BEST ✅
Epoch   7/20 | loss=0.3026 acc=0.9483 | val_loss=0.3176 val_acc=0.9018 ← BEST ✅
Epoch   8/20 | loss=0.2796 acc=0.9626 | val_loss=0.3062 val_acc=0.9020 ← BEST ✅
Epoch   9/20 | loss=0.2518 acc=0.9709 | val_loss=0.2188 val_acc=0.9436 ← BEST ✅
Epoch  10/20 | loss=0.2386 acc=0.9774 | val_loss=0.1998 val_acc=0.9524 ← BEST ✅
Epoch  11/20 | loss=0.2312 acc=0.9823 | val_loss=0.1652 val_acc=0.9704 ← BEST ✅
Epoch  12/20 | loss=0.2066 acc=0.9847 | val_loss=0.1626 val_acc=0.9716 ← BEST ✅
Epoch  13/20 | loss=0.2181 acc=0.9865 | 

In [8]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# CELL 5: EVALUASI + THRESHOLD TUNING
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
print("\n" + "="*60)
print("CELL 5 — Final Evaluation + Threshold Tuning")
print("="*60)

model.load_weights("best_weights_v3.weights.h5")

y_true_list, y_pred_list = [], []
for (cv_b, job_b), y_b in val_ds:
    preds = model([cv_b, job_b], training=False)
    y_true_list.extend(y_b.numpy())
    y_pred_list.extend(preds.numpy().flatten())

y_true       = np.array(y_true_list)
y_pred_probs = np.array(y_pred_list)

# ── Threshold tuning via F1 maximization ─────────────────────────────
thresholds = np.arange(0.3, 0.71, 0.01)
best_thr, best_f1_acc = 0.5, 0.0
for thr in thresholds:
    y_bin = (y_pred_probs >= thr).astype(int)
    acc   = accuracy_score(y_true, y_bin)
    if acc > best_f1_acc:
        best_f1_acc = acc
        best_thr    = thr

y_pred_binary = (y_pred_probs >= best_thr).astype(int)
final_acc     = accuracy_score(y_true, y_pred_binary)
auc_score     = roc_auc_score(y_true, y_pred_probs)

print(f"\n{'='*60}")
print(f"  Best Threshold (acc-optimal): {best_thr:.2f}")
print(f"  BEST Val Accuracy (epoch)   : {best_val_acc*100:.2f}%")
print(f"  Final Val Accuracy          : {final_acc*100:.2f}%")
print(f"  ROC-AUC Score               : {auc_score:.4f}")
print(f"{'='*60}")
print(classification_report(y_true, y_pred_binary,
                             target_names=["Not Match (0)", "Match (1)"]))

if final_acc >= 0.92:
    print("🎯 TARGET >92% TERCAPAI! ✅")
else:
    gap = (0.92 - final_acc) * 100
    print(f"⚠  Masih {gap:.2f}% dari target. Lihat catatan tuning di bawah.")

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# CELL 6: SAVE
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
print("\n" + "="*60)
print("CELL 6 — Saving Model & Vocabulary")
print("="*60)

model.save("neo_karir_job_match_v3.keras", overwrite=True, save_format="keras")

vocab_payload = {
    "vocabulary"             : vectorizer.get_vocabulary(),
    "max_tokens"             : MAX_TOKENS,
    "output_mode"            : "int",
    "output_sequence_length" : SEQUENCE_LENGTH,
    "standardize"            : "lower_and_strip_punctuation",
    "best_threshold"         : best_thr,
}
with open("neo_karir_vocab_v3.pkl", "wb") as f:
    pickle.dump(vocab_payload, f)

print("✅ Model saved  : neo_karir_job_match_v3.keras")
print("✅ Vocab saved  : neo_karir_vocab_v3.pkl")
print(f"   (threshold optimal disimpan di vocab: {best_thr:.2f})")

print(f"\n🎯 Target >92%  → {'✅ ACHIEVED' if final_acc >= 0.92 else '⚠ Belum — coba tuning notes di bawah'}")


CELL 5 — Final Evaluation + Threshold Tuning



  Best Threshold (acc-optimal): 0.68
  BEST Val Accuracy (epoch)   : 97.91%
  Final Val Accuracy          : 98.22%
  ROC-AUC Score               : 0.9984
               precision    recall  f1-score   support

Not Match (0)       1.00      0.98      0.99      2928
    Match (1)       0.96      0.99      0.97      1572

     accuracy                           0.98      4500
    macro avg       0.98      0.98      0.98      4500
 weighted avg       0.98      0.98      0.98      4500

🎯 TARGET >92% TERCAPAI! ✅

CELL 6 — Saving Model & Vocabulary
✅ Model saved  : neo_karir_job_match_v3.keras
✅ Vocab saved  : neo_karir_vocab_v3.pkl
   (threshold optimal disimpan di vocab: 0.68)

🎯 Target >92%  → ✅ ACHIEVED
